# SV2 follow-up — **arm W0D** (motion dropout alone) + **SA²-v2 probe**, seed 2024

Rule (written before any number): `core/docs/final/SA2V2_RUN.md` §8 (W0D) and §9 (probe). Upload branch
`v_final_stp_v2` to `Thesis-Final/kat-vad`. Needs the SV2 run and its reads from `sa2v2_nexar_dada.ipynb`
(`outputs/sv2/SV2/s2024`, `outputs/sv2/eval/{dada,dota}/SV2`).

| step | what | GPU | estimate |
|---|---|---|---|
| 1–3 | setup, preflight (same as the SV2 notebook) | — | 5–10 min |
| 4 | pytest on the new code (W0D, probe) — **stop if red** | no | 2–5 min |
| 5 | probe `adam`: SV2's AdamW state | no | 2–5 min |
| 6 | probe `live`: per-term gradients at init and at the final checkpoint | **yes** | 5–10 min |
| 7 | train W0D (1,749 steps) | **yes** | 15–45 min |
| 8 | W0D: DADA read + DoTA-CAP-dev read | **yes** | 10–20 min |
| 9 | read-out → **paste `REPORTS/w0d_readout.md` + `REPORTS/probe/*.md` to Claude** | — | |

Estimates only. Every step skips what is already on Drive. One seed ⇒ **PRELIMINARY (n = 1 seed)**.


In [ ]:
from google.colab import drive

drive.mount("/content/drive")

In [ ]:
import json
import os
import sys
from pathlib import Path

DRIVE    = '/content/drive/MyDrive/Thesis'        # read-only: data, cache/clip
DRIVE_V2 = '/content/drive/MyDrive/Thesis-V2'     # read-only: Nexar_window, A2 bakes, KNN, v2 A2 runs
DRIVE_V5 = '/content/drive/MyDrive/Thesis-V5'     # read-only: 14 x 14 CLIP token grids (branch v5-r7)
DRIVE_F  = '/content/drive/MyDrive/Thesis-Final'  # code + every cache / output this notebook writes
os.environ['PROJECT_ROOT']       = DRIVE_F
os.environ['KATVAD_DATA_ROOT']   = f'{DRIVE}/data'
os.environ['KATVAD_CACHE_ROOT']  = f'{DRIVE_F}/cache'
os.environ['KATVAD_CKPT_ROOT']   = f'{DRIVE}/ckpts'
os.environ['KATVAD_OUTPUT_ROOT'] = f'{DRIVE_F}/outputs'

os.environ['REPO'] = f'{DRIVE_F}/kat-vad'
REPO = Path(os.environ['REPO'])
assert (REPO / 'core' / 'constants.py').is_file(), f'no code at {REPO} -- upload branch v_final_stp_v2 there'
os.environ['PYTHONPATH'] = str(REPO)
if str(REPO) not in sys.path:
    sys.path.insert(0, str(REPO))
for name in [m for m in sys.modules if m == 'core' or m.startswith('core.')]:
    del sys.modules[name]              # constants reads KATVAD_* at import time
from core import constants  # noqa: E402

assert constants.REPO_ROOT == REPO, f'imported core from {constants.REPO_ROOT}, expected {REPO}'
assert hasattr(constants, 'W0D_ARM'), 'this upload predates W0D / the probe -- re-upload kat-vad/'
assert '## 8. Arm W0D' in (REPO / 'core/docs/final/SA2V2_RUN.md').read_text(), 'SA2V2_RUN.md has no §8 (W0D rule) in this upload'

SEED, ARM, SHUF = constants.SV2_SEED, constants.SV2_ARM, constants.SV2_SHUFFLED_ARM
W0D = constants.W0D_ARM
ENC      = constants.V5_ENCODER
C        = constants.NEXAR_CONSTRUCTION_WINDOW
DATASET  = constants.NEXAR_DATASET
T2NAME   = constants.DADA_ORIGIN_DATASET
CAP_DEV  = constants.V2_SPLIT_DOTA_CAP_DEV
S_N, S_DADA, S_DOTA = constants.STP_NEXAR_STRIDE, constants.ZS_STRIDE, constants.STP_DOTA_STRIDE

V2_CACHE, V2_OUT = Path(DRIVE_V2) / 'cache', Path(DRIVE_V2) / 'outputs'
BUILD    = V2_OUT / 'v2' / 'REPORTS' / 'nexar_n4' / 'build_report.json'
DATA     = Path(DRIVE_V2) / 'data' / f'{DATASET}_{C}'
A2_NEXAR = V2_CACHE / 'v2' / 'A2_S' / 'Nexar_s8'
A2_DADA  = V2_CACHE / 'v2' / 'A2_S' / f'{T2NAME}_test_sources_nexar'
A2_DOTA  = V2_CACHE / 'v2' / 'A2_S' / f'{CAP_DEV}_s3_from_s1_nexar'
KNN      = V2_CACHE / 'knn' / f'{DATASET}_{C}' / constants.KNN_CACHE_FILENAME
A2_RUNS  = V2_OUT / constants.NEXAR_FULL_RUNS_DIR / 'A2'
GRIDS    = Path(DRIVE_V5) / 'cache' / 'video_grid' / 'clip_vit_b16'
GRID_N   = GRIDS / f'{DATASET}_s{S_N}_cgrid14'
GRID_D   = GRIDS / f'{constants.DOTA_CAP_DATASET}_s{S_DOTA}_cgrid14'
T2       = constants.DATA_ROOT / T2NAME
T2_CLIP  = Path(DRIVE) / 'cache' / 'clip' / T2NAME
T2_CENSUS = T2 / 'counts' / 'resolved_census.json'
ANNOT    = constants.DATA_ROOT / 'DADA2000Origin' / constants.DADA_ORIGIN_ROOT_DIRNAME / constants.DADA_ORIGIN_ANNOTATION_FILENAME
ARC_DADA = Path(DRIVE_F) / 'data' / constants.ZS_FRAMES_DIRNAME / T2NAME
DOTA     = constants.DATA_ROOT / 'DoTA'
DOTA_S1  = Path(DRIVE) / 'cache' / 'clip' / 'DoTA_s1_ncc'
ZS_ROOT  = constants.OUTPUT_ROOT / constants.ZS_OUTPUT_DIRNAME
W0_RUN   = constants.OUTPUT_ROOT / constants.WU_OUTPUT_DIRNAME / constants.ZS_CONTROL_ARM / f's{SEED}'
W0_DADA  = ZS_ROOT / 'eval' / 'dada' / constants.ZS_CONTROL_ARM
R4_DADA  = ZS_ROOT / 'eval' / 'dada' / constants.ZS_SA2_ARM
W0_DOTA  = constants.OUTPUT_ROOT / constants.WU_OUTPUT_DIRNAME / 'eval' / 'dota' / CAP_DEV / constants.ZS_CONTROL_ARM

from core.data import stp_patches as stp_data      # noqa: E402

CELL_N, CELL_DADA = stp_data.cell_cache_dir(DATASET, S_N), stp_data.cell_cache_dir(T2NAME, S_DADA)
CELL_DOTA = stp_data.cell_cache_dir(constants.DOTA_CAP_DATASET, S_DOTA)
ROOT     = constants.OUTPUT_ROOT / constants.SV2_OUTPUT_DIRNAME
REPORTS  = ROOT / 'REPORTS'
RUN_DIR  = {ARM: ROOT / ARM / f's{SEED}', SHUF: ROOT / SHUF / f's{SEED}', W0D: ROOT / W0D / f's{SEED}'}
PROBE    = REPORTS / constants.SV2_PROBE_DIRNAME
LOCAL    = Path('/content/sv2')
for p in (REPORTS, LOCAL):
    p.mkdir(parents=True, exist_ok=True)

PLAN = json.loads(BUILD.read_text())['plan'][C]
STEPS, EPOCHS = PLAN['steps'], PLAN['epochs']     # 1,749 = 33 x 53
for name, path in (('Nexar_window', DATA), ('Nexar A2 bake', A2_NEXAR), ('DADA A2 bake', A2_DADA), ('KNN', KNN),
                   ('v2 A2 runs', A2_RUNS), ('Nexar tokens', GRID_N), ('T2 meta', T2 / constants.META_FILENAME),
                   ('T2 CLIP', T2_CLIP), ('T2 census', T2_CENSUS), ('DADA annotation', ANNOT),
                   ('DADA frames', ARC_DADA / constants.ZS_FRAMES_MANIFEST), ('W0 run', W0_RUN / 'metrics.jsonl'),
                   ('W0 DADA read', W0_DADA / 'clip_aucs.json')):
    assert path.exists(), f'missing {name}: {path}'
    print(f'  {name:16s} {path}')
print(f'steps {STEPS} ({EPOCHS} epochs) | cells -> {CELL_N} | {CELL_DADA}\nruns/reads -> {ROOT}')
for name, path in (('SV2 run', RUN_DIR[ARM] / 'checkpoint_last.pt'), ('SV2 DADA read', ROOT / 'eval' / 'dada' / ARM / 'clip_aucs.json'),
                   ('SV2 DoTA read', ROOT / 'eval' / 'dota' / ARM / 'clip_aucs.json'), ('W0 DoTA read', W0_DOTA / 'clip_aucs.json'),
                   ('Nexar cells', CELL_N / constants.SV2_LEVEL_FILENAME)):
    assert path.exists(), f'missing {name}: {path} -- run sa2v2_nexar_dada.ipynb first'
print('SV2 run + reads found')


In [ ]:
%%bash
pip install -q "transformers==4.56.*" av einops faiss-cpu scikit-learn openpyxl xlrd
python -c "import av, einops, faiss, sklearn, openpyxl; print('deps OK')"
nvidia-smi --query-gpu=name,memory.total --format=csv,noheader || echo 'NO GPU -- steps 5-7, 9, 10 need one'
df -h /content | tail -1

In [ ]:
import pickle
import shutil
import subprocess
import threading
import time

import numpy as np
import torch
import yaml

from core.tools.dada_whole_eval import test_sources    # noqa: E402
from core.tools.sa2v2_cells import training_sources    # noqa: E402
from core.tools.v5_stage1 import _flat                 # noqa: E402

ENV = dict(os.environ, USE_TF='0', TF_CPP_MIN_LOG_LEVEL='2')   # transformers must not import TF (absl eats INFO logs)


def run(cmd, log=None):
    """Stream a child's output here (and to ``log``); raise with its tail on failure."""
    tail = []
    sink = open(log, 'a') if log else None
    try:
        with subprocess.Popen([str(c) for c in cmd], cwd=str(REPO), env=ENV, stdout=subprocess.PIPE,
                              stderr=subprocess.STDOUT, text=True, bufsize=1) as proc:
            for line in proc.stdout:
                tail = (tail + [line])[-40:]
                if sink:
                    sink.write(line)
                print(line, end='')
    finally:
        if sink:
            sink.close()
    if proc.returncode:
        print('\n' + '=' * 30 + f'\nFAILED (exit {proc.returncode}) -- paste these lines to Claude:\n' + ''.join(tail))
        raise RuntimeError(f'exit {proc.returncode}: {" ".join(str(c) for c in cmd[:4])} ...')


def write_ids(path, ids):
    path.parent.mkdir(parents=True, exist_ok=True)
    path.write_text(''.join(f'{v}\n' for v in ids))
    return path


def stage(src, dst, ids=None):
    """Copy a Drive dir (only ``ids`` + json manifests if given) or file to VM disk once (bit-identical; C2)."""
    t0 = time.time()
    if src.is_file():
        dst.parent.mkdir(parents=True, exist_ok=True)
        if not dst.exists():
            shutil.copy2(src, dst)
        return dst
    dst.mkdir(parents=True, exist_ok=True)
    if ids is None:
        for f in src.iterdir():
            if f.is_file() and not (dst / f.name).exists():
                shutil.copy2(f, dst / f.name)
    else:
        for v in ids:
            if not (dst / f'{v}.npy').exists():
                shutil.copy2(src / f'{v}.npy', dst / f'{v}.npy')
        for pattern in ('*.json', '*.npz'):
            for m in src.glob(pattern):
                shutil.copy2(m, dst / m.name)
    print(f'  staged {src.name} in {time.time() - t0:.0f}s')
    return dst


def publish(src, dst):
    """VM-local result -> Drive ATOMICALLY: ``<dst>.partial``, fsync, rename (C10)."""
    if src.is_dir():
        for f in [p for p in src.rglob('*') if p.is_file()]:
            publish(f, dst / f.relative_to(src))
        return
    dst.parent.mkdir(parents=True, exist_ok=True)
    tmp = dst.with_name(dst.name + '.partial')
    shutil.copy2(src, tmp)
    with open(tmp, 'rb+') as fh:
        os.fsync(fh.fileno())
    os.replace(tmp, dst)


def flush_drive():
    t0 = time.time()
    drive.flush_and_unmount()
    drive.mount('/content/drive')
    print(f'Drive flushed and remounted in {time.time() - t0:.0f}s')


def rows(d, v):
    return int(np.load(d / f'{v}.npy', mmap_mode='r').shape[0])


def ckpt_step(path):
    """The checkpoint's own global_step; 0 if absent, -1 if unreadable (never equals a real step)."""
    if not path.exists():
        return 0
    try:
        return int(torch.load(path, map_location='cpu', weights_only=False).get('global_step', 0))
    except (RuntimeError, EOFError, pickle.UnpicklingError) as exc:
        print(f'UNREADABLE {path}: {exc}')
        return -1


def metrics_last(path):
    lines = [l for l in path.read_text().splitlines() if l.strip()] if path.exists() else []
    return json.loads(lines[-1]) if lines else None


def is_verified(drive_dir):
    rec, ckpt = drive_dir / 'verified.json', drive_dir / 'checkpoint_last.pt'
    if rec.exists() and ckpt.exists():
        r = json.loads(rec.read_text())
        return r['step'] == STEPS and r['ckpt_bytes'] == ckpt.stat().st_size
    return False


class GpuPeak:
    """Peak GPU memory of a block, sampled with nvidia-smi (proposal analysis 6: cost)."""

    def __enter__(self):
        self.peak, self.stop = 0, threading.Event()
        self.thread = threading.Thread(target=self._loop, daemon=True)
        self.thread.start()
        return self

    def _loop(self):
        while not self.stop.wait(5):
            out = subprocess.run(['nvidia-smi', '--query-gpu=memory.used', '--format=csv,noheader,nounits'],
                                 capture_output=True, text=True).stdout.split()
            if out:
                self.peak = max(self.peak, int(out[0]))

    def __exit__(self, *exc):
        self.stop.set()
        self.thread.join()
        print(f'peak GPU memory: {self.peak} MiB')


# ids and the preflight: nothing below runs on a wrong corpus
NEXAR_IDS = training_sources(DATA)
DADA_IDS = test_sources(T2)
assert len(NEXAR_IDS) == 451, f'{len(NEXAR_IDS)} Nexar training sources, expected 451'
assert len(DADA_IDS) == 370, len(DADA_IDS)
miss = [v for v in NEXAR_IDS if not (GRID_N / f'{v}.npy').exists()]
assert not miss, f'{len(miss)} Nexar sources have no token grid in {GRID_N}: {miss[:5]}'
miss = [v for v in DADA_IDS if not (A2_DADA / f'{v}.npy').exists()]
assert not miss, f'{len(miss)} DADA sources have no A2 bake: {miss[:5]}'
g = np.load(GRID_N / f'{NEXAR_IDS[0]}.npy', mmap_mode='r')
assert g.size == g.shape[0] * 14 * 14 * 512, f'token grid has shape {g.shape}, expected (L, 14, 14, 512)'
man = json.loads((ARC_DADA / constants.ZS_FRAMES_MANIFEST).read_text())
assert man['pass'] and man['stride'] == S_DADA, man
for name, d in (('Nexar', A2_NEXAR), ('DADA', A2_DADA)):
    m = json.loads((d / constants.V2_INPUT_MANIFEST_FILENAME).read_text())
    assert (m['crn'], m['motion']) == (constants.V2_OFF, ENC), f'{name} bake is {m}'
same = {json.loads((d / constants.V2_INPUT_MANIFEST_FILENAME).read_text())['train_ids_sha1'] for d in (A2_NEXAR, A2_DADA)}
assert len(same) == 1, f'the bakes carry different statistics: {same}'
ID_N, ID_DADA = write_ids(LOCAL / 'nexar.txt', NEXAR_IDS), write_ids(LOCAL / 'dada.txt', DADA_IDS)
L_DEFS = LOCAL / 'nexar_defs'
L_DEFS.mkdir(parents=True, exist_ok=True)
for f in DATA.glob('*.json'):               # defs.json etc.: the definitions the arms trained with
    shutil.copy2(f, L_DEFS / f.name)
print(f'preflight OK: {len(NEXAR_IDS)} Nexar sources with tokens, {len(DADA_IDS)} DADA sources, shared statistics')

## 4. Tests on the new code (CPU) — **stop and paste the output if anything fails**


In [ ]:
%%bash
cd "$REPO"
python -m pytest -q -p no:cacheprovider core/tests/test_w0d_probe.py core/tests/test_sa2v2.py core/tests/test_wu_arms.py 2>&1 | tail -15


## 5. Probe `adam` (CPU): SV2's AdamW state (§9)

Needs CLIP's text tower (HF download): the init is rebuilt as `core.train` builds it.


In [ ]:
t0 = time.time()
if not (PROBE / 'sv2_probe_adam.md').exists():
    run([sys.executable, '-m', 'core.tools.sv2_probe', 'adam', '--run-dir', RUN_DIR[ARM], '--out-dir', LOCAL / 'probe'],
        log=LOCAL / 'probe_adam.log')
    publish(LOCAL / 'probe', PROBE)
print((PROBE / 'sv2_probe_adam.md').read_text())
print(f'probe adam in {(time.time() - t0) / 60:.1f} min')


## 6. Probe `live` (GPU): per-term gradients, at init and at the final checkpoint (§9)


In [ ]:
assert torch.cuda.is_available(), 'torch sees no GPU'
t0 = time.time()
data_local = stage(DATA, LOCAL / 'stage' / 'data')
knn_local = stage(KNN, LOCAL / 'stage' / 'knn.npz')
a2_local = stage(A2_NEXAR, LOCAL / 'stage' / 'a2_nexar', NEXAR_IDS)
cell_local = stage(CELL_N, LOCAL / 'stage' / 'cell_nexar', NEXAR_IDS)
for name, ckpt in (('init', None), ('final', RUN_DIR[ARM] / 'checkpoint_last.pt')):
    out = PROBE / f'live_{name}'
    if not (out / 'sv2_probe_live.md').exists():
        local = LOCAL / 'probe' / f'live_{name}'
        run([sys.executable, '-m', 'core.tools.sv2_probe', 'live', '--config', RUN_DIR[ARM] / 'config.yaml',
             '--name', name, *(['--checkpoint', ckpt] if ckpt else []), '--data-dir', data_local,
             '--clip-dir', a2_local, '--patch-dir', cell_local, '--knn-cache', knn_local, '--out-dir', local],
            log=LOCAL / f'probe_live_{name}.log')
        publish(local, out)
    print((out / 'sv2_probe_live.md').read_text())
flush_drive()
print(f'probe live in {(time.time() - t0) / 60:.1f} min')


## 7. Train W0D (GPU): W0 + motion dropout 0.3, no spatial module (§8)

**Watch the log: past 2 s/step, stop and paste it to Claude.**


In [ ]:
def w0d_cmd(out_dir, data, knn):
    cmd = [sys.executable, '-m', 'core.train',
           '--set', 'train.stage=2', '--set', f'train.seed={SEED}',
           '--set', f'train.num_epochs={EPOCHS}', '--set', 'train.amp=true',
           '--set', f'data.dataset={DATASET}',
           '--set', f'model.score_head_kernel={constants.DADA_ORIGIN_SCORE_HEAD_KERNEL}',
           '--set', f'loss.mil_topk_pct={constants.DADA_ORIGIN_MIL_TOPK_PCT}',
           '--set', 'kip.enabled=false', '--set', 'loss.dvs_anchor_mode=span',
           '--set', 'data.is_egocentric=false',
           '--set', f'v2.motion={ENC}', '--set', f'model.motion_dim={constants.V5_GRID_DIM}',
           '--set', f'model.motion_dropout={constants.W0D_MOTION_DROPOUT}',
           '--data-dir', data, '--clip-dir', stage(A2_NEXAR, LOCAL / 'stage' / 'a2_nexar', NEXAR_IDS),
           '--output-dir', out_dir]
    return cmd + (['--knn-cache', knn] if knn else [])


def train_w0d(data, knn):
    drive_dir, local = RUN_DIR[W0D], LOCAL / 'runs' / W0D
    if is_verified(drive_dir):
        print(f'{W0D} s{SEED}: verified on Drive -- not retrained')
        return
    assert torch.cuda.is_available(), 'torch sees no GPU -- train.device=auto would silently train on CPU'
    local.mkdir(parents=True, exist_ok=True)
    resume = ['--resume', local / 'checkpoint_last.pt'] if (local / 'checkpoint_last.pt').exists() else []
    t0 = time.time()
    with GpuPeak():
        run(w0d_cmd(local, data, knn) + resume, log=LOCAL / f'train_{W0D}.log')
    last = metrics_last(local / 'metrics.jsonl')
    assert ckpt_step(local / 'checkpoint_last.pt') == last['global_step'] == STEPS, f'{W0D}: J10 locally'
    for name in ('checkpoint_last.pt', 'metrics.jsonl', 'config.yaml'):
        publish(local / name, drive_dir / name)
    flush_drive()
    assert ckpt_step(drive_dir / 'checkpoint_last.pt') == STEPS, f'{W0D}: Drive copy is not step {STEPS}'
    (LOCAL / 'verified_w0d.json').write_text(json.dumps(
        {'step': STEPS, 'ckpt_bytes': (drive_dir / 'checkpoint_last.pt').stat().st_size}))
    publish(LOCAL / 'verified_w0d.json', drive_dir / 'verified.json')
    print(f'{W0D}: {(time.time() - t0) / 60:.0f} min | motion_share {last.get("motion_share")} | w_u_norm {last.get("w_u_norm")}')


def check_w0d():
    d = RUN_DIR[W0D]
    payload = torch.load(d / 'checkpoint_last.pt', map_location='cpu', weights_only=False)
    assert payload['global_step'] == metrics_last(d / 'metrics.jsonl')['global_step'] == STEPS, f'{W0D}: J10'
    assert not any(k.startswith('sa2v2.') for k in payload['model']), f'{W0D}: has SA²-v2 weights'
    ref = _flat(yaml.safe_load((A2_RUNS / f's{SEED}' / 'config.yaml').read_text()))
    got = _flat(yaml.safe_load((d / 'config.yaml').read_text()))
    diff = {k: (ref.get(k), got.get(k)) for k in set(ref) | set(got)
            if not k.startswith(('stp.', 'v5.')) and k not in ('model.motion_init_std', 'train.motion_lr_mult')
            and ref.get(k) != got.get(k)}
    assert set(diff) == {'model.motion_dropout'}, f'{W0D} differs from v2 A2 s{SEED} beyond model.motion_dropout: {diff}'
    assert got['model.motion_dropout'] == constants.W0D_MOTION_DROPOUT and got['stp.sa2v2'] is False and got['stp.sa2'] is False
    assert got['stp.text_prompt'] == constants.STP_PROMPT_RELEASED, f'{W0D}: prefix is not OFF'
    print(f'{W0D}: J10 {STEPS}, config == v2 A2 s{SEED} + model.motion_dropout={constants.W0D_MOTION_DROPOUT}, no SA²-v2')


train_w0d(stage(DATA, LOCAL / 'stage' / 'data'), stage(KNN, LOCAL / 'stage' / 'knn.npz'))
check_w0d()


## 8. W0D reads (GPU): DADA-2000 whole video + DoTA-CAP-dev protocol B, W0's own protocols


In [ ]:
from core.data.v2_splits import load_split    # noqa: E402

assert torch.cuda.is_available(), 'torch sees no GPU'
CKPT_W0D = RUN_DIR[W0D] / 'checkpoint_last.pt'
L_T2 = LOCAL / 'T2_meta'
L_T2.mkdir(parents=True, exist_ok=True)
shutil.copy2(T2 / constants.META_FILENAME, L_T2 / constants.META_FILENAME)
DADA_OUT, DOTA_OUT = ROOT / 'eval' / 'dada', ROOT / 'eval' / 'dota'
if not (DADA_OUT / W0D / 'clip_aucs.json').exists():       # W0's protocol: A2 bake, no --patch-dir
    local = LOCAL / 'dada' / W0D
    run([sys.executable, '-m', 'core.tools.dada_whole_eval', '--run', f's{SEED}', CKPT_W0D, '--t2-dir', L_T2,
         '--annotation', ANNOT, '--census', T2_CENSUS, '--input-dir', stage(A2_DADA, LOCAL / 'bake_dada', DADA_IDS),
         '--data-dir', L_DEFS, '--out-dir', local], log=LOCAL / 'dada_w0d.log')
    publish(local, DADA_OUT / W0D)
r = json.loads((DADA_OUT / W0D / 'dada_readout.json').read_text())
print(f"dada {W0D}: macro {r['macro']['mean']:.4f} over {r['two_class']} sources | micro {r['micro']:.4f}")

if not (DOTA_OUT / W0D / 'clip_aucs.json').exists():       # protocol B, as W0's stored read
    CAP_IDS = sorted(load_split(CAP_DEV))
    local = LOCAL / 'dota' / W0D
    run([sys.executable, '-m', 'core.tools.protocol_b_eval', '--run', f's{SEED}', CKPT_W0D,
         '--input-dir', stage(A2_DOTA, LOCAL / 'bake_dota', CAP_IDS), '--s1-dir', stage(DOTA_S1, LOCAL / 'dota_s1', CAP_IDS),
         '--metadata', DOTA / 'metadata_val.json', '--split-file', DOTA / 'val_split.txt',
         '--data-dir', stage(DOTA / 'labels_s8', LOCAL / 'dota_labels'), '--split', CAP_DEV, '--out-dir', local,
         '--device', 'auto'], log=LOCAL / 'dota_w0d.log')
    publish(local, DOTA_OUT / W0D)
r = json.loads((DOTA_OUT / W0D / 'protocol_b_readout.json').read_text())
print(f"dota {W0D}: macro {r['macro']['mean']:.4f}")
flush_drive()


## 9. Read-out → **paste `REPORTS/w0d_readout.md` and the three probe `.md` files to Claude**


In [ ]:
run([sys.executable, '-m', 'core.tools.w0d_readout',
     '--w0-dada', W0_DADA, '--w0d-dada', ROOT / 'eval' / 'dada' / W0D, '--sv2-dada', ROOT / 'eval' / 'dada' / ARM,
     '--w0-dota', W0_DOTA, '--w0d-dota', ROOT / 'eval' / 'dota' / W0D, '--sv2-dota', ROOT / 'eval' / 'dota' / ARM,
     '--out-dir', REPORTS])
print((REPORTS / 'w0d_readout.md').read_text())
drive.flush_and_unmount()          # C10: every pending Drive upload lands before the VM goes
drive.mount('/content/drive')
print('Drive flushed -- paste REPORTS/w0d_readout.md and REPORTS/probe/{sv2_probe_adam.md, live_init/, live_final/} to Claude')
